# 삼성전자 4Q25 — Multi-Query RAG (OpenAI 버전)

**기존 HuggingFace 버전에서 변경된 부분:**

| 항목 | 기존 (HuggingFace) | 변경 (OpenAI) |
|---|---|---|
| 임베딩 | BAAI/bge-m3 | text-embedding-3-small |
| LLM | google/gemma-2-9b-it | gpt-4o-mini |
| 벡터스토어 | FAISS | Chroma (OpenAI와 궁합 좋음) |
| GPU 필요 | ✅ | ❌ (API 호출이라 불필요) |

**나머지 (RAG 체인, 프롬프트 구조, 파이프라인)는 동일**

## 0. OpenAI API 키 설정

처음 사용하는 경우 아래 순서로 진행:
1. https://platform.openai.com 접속
2. 회원가입 → API Keys → Create new secret key
3. 발급받은 키를 `.env` 파일에 추가:
```
OPENAI_API_KEY=sk-...
```

필요한 패키지 설치 (처음 한 번만):
```bash
pip install langchain-openai chromadb
```

## 1. 환경 설정 및 라이브러리 임포트

In [1]:
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI   # ← HuggingFace에서 변경
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.load import dumps, loads
from operator import itemgetter

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

# API 키 확인
if openai_api_key:
    print(f"✅ OpenAI API 키 로드 완료: sk-...{openai_api_key[-4:]}")
else:
    print("❌ API 키가 없습니다. .env 파일을 확인하세요.")

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ OpenAI API 키 로드 완료: sk-...yK4A


In [2]:
file_paths = [
    "C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf",
    "C:/Users/user/Downloads/삼성_2025Q4_script_eng_AudioScript.pdf"
]

docs = []

for path in file_paths:
    # 각 파일을 로드
    loader = PyPDFLoader(path)
    
    # load를 통해 문서의 각 페이지를 Document 객체로 변환
    docs.extend(loader.load())

print(f"총 로드된 페이지 수: {len(docs)}")
# AudioSciprt의 페이지 수: 34
# presentation의 페이지 수: 15
# 총합 로드 페이지 수: 49

총 로드된 페이지 수: 49


In [3]:
# 확인용 — 어느 파일에서 왔는지 출력
for doc in docs[:5]:
    print(doc.metadata.get("source"), "|", doc.metadata.get("page_number"))

print(docs[0])
print(docs[1])

C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf | None
C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf | None
C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf | None
C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf | None
C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf | None
page_content='SAMSUNG 
ELECTRONICS
Earnings Presentation: 
4Q 2025 Financial Results' metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-01-28T20:17:53+09:00', 'title': 'Samsung  Electronics', 'author': '권해주/커뮤니케이션컨텐츠그룹/Senior Professional/삼성전자', 'moddate': '2026-01-28T20:17:53+09:00', 'source': 'C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}
page_content='The financial information in this document are consolidated earnings results based on K-IFRS.
This document is provided for the convenience 

## 3. Text Splitting

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,    
    chunk_overlap=220
)

splits = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(splits)}\n")
print(f"첫번째 청크 예시:\n{splits[0].page_content}\n")
print(f"두번째 청크 예시:\n{splits[1].page_content}")

분할된 청크 수: 78

첫번째 청크 예시:
SAMSUNG 
ELECTRONICS
Earnings Presentation: 
4Q 2025 Financial Results

두번째 청크 예시:
The financial information in this document are consolidated earnings results based on K-IFRS.
This document is provided for the convenience of investors only before the external audit on our 4Q 2025 financial results iscompleted. 
The Audit outcomes may cause some parts of this document to change. 
This document contains "forward-looking statements" - that is statements related to future not past events. 
In this context "forward-looking statements" often address our expected future business and financial performance 
and often contain words such as "expects" "anticipates" "intends" "plans" "believes" "seeks" or "will". 
"Forward-looking statements" by their nature address matters that are to different degrees uncertain. 
For us particular uncertainties which could adversely or positively affect our future results include:
·  The behavior of financial markets including fluctuatio

## 4. Vector Store

**변경점:**
- 임베딩: `BAAI/bge-m3` → `text-embedding-3-small`
- 벡터스토어: `FAISS` → `Chroma`
- GPU 불필요 (API 호출 방식)

**에러 발생시**
https://developers.openai.com/api/docs/guides/error-codes#api-errors 을 통해 에러 확인: 높은 확률로 Billing set-up 안되어 있기 때문

429 에러: 빌링 이슈

https://platform.openai.com/settings/organization/billing/overview 에서 빌링 업데이트 필수

https://platform.openai.com/settings/organization/limits 리밋 확대 URL


In [5]:
# OpenAI 임베딩 (API 호출 방식 — GPU 불필요)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

# Chroma 벡터스토어
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

# Multi-Query: 쿼리당 k=4 (5쿼리 × 4 = 최대 20개 → 중복 제거)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Vector store 준비 완료")

Vector store 준비 완료


## 5. LLM 설정

**변경점:** `gemma-2-9b-it` → `gpt-4o-mini`

gpt-4o-mini를 선택한 이유:
- 수치 추론 및 테이블 해석 능력이 Gemma-9b 대비 훨씬 뛰어남
- 불확실한 컨텍스트에서 hallucination 대신 "모르겠다"고 답하는 경향
- 비용: 질문 1번당 약 $0.0005 (약 0.7원)

In [6]:
chat_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=openai_api_key
)

print("LLM 준비 완료: gpt-4o-mini")

LLM 준비 완료: gpt-4o-mini


## 6. Multi-Query RAG 구성

---여기서부터는 HuggingFace 버전과 완전히 동일---

In [7]:
# Step 1: Multi-Query 생성 체인
# 사용자 질문 1개 → LLM이 5가지 다른 관점으로 재작성
QUERY_PROMPT = ChatPromptTemplate.from_template("""
You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. 

Original question: {question}
""")

query_chain = (
    QUERY_PROMPT
    | chat_llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

print("Multi-Query 체인 준비 완료")

Multi-Query 체인 준비 완료


In [8]:
# Step 2: Unique Union
# 5개의 검색 결과를 합치고 중복 제거
def get_unique_union(documents: list[list]):
    """
    5개 쿼리로 검색된 문서 리스트를 받아 중복을 제거하고 반환.
    
    Args:
        documents: [[doc1, doc2], [doc2, doc3], ...] 형태
    Returns:
        중복이 제거된 unique Document 리스트
    """
    # 중첩 리스트 flatten + Document → JSON string 변환 (set 비교용)
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # 중복 제거
    unique_docs = list(set(flattened_docs))
    # 다시 Document 객체로 복원
    return [loads(doc) for doc in unique_docs]

In [9]:
# retrieval_chain: 질문 → 5개 재작성 → 병렬 검색 → 중복 제거
retrieval_chain = query_chain | retriever.map() | get_unique_union

## 7. 최종 RAG 체인 구성

In [10]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an expert AI assistant specializing in Samsung Electronics earnings reports.
Answer the question based ONLY on the provided context.
If the context does not contain enough information to answer confidently,
say "The provided context does not contain sufficient information on this topic."
Always include specific numbers and financial figures when available.

#Context:
{context}

#Question:
{question}

#Answer:
""")

final_rag_chain = (
    {
        "context": retrieval_chain,
        "question": itemgetter("question")
    }
    | RAG_PROMPT
    | chat_llm
    | StrOutputParser()
)

print("최종 RAG 체인 준비 완료")

최종 RAG 체인 준비 완료


## 8. 실행 및 테스트

In [11]:
from langchain_teddynote.messages import stream_response

In [12]:
# 테스트 1: 전반적인 실적
question = "삼성전자 2025 4분기 실적 발표에 대해서 알려줘"
answer = final_rag_chain.invoke({"question": question})

stream_response(answer)
# Langsmith Trace
# https://smith.langchain.com/public/08a89292-7eaa-4476-8549-109bfd2f2b4b/r

C:\Users\user\AppData\Local\Temp\ipykernel_7496\3162270408.py:17: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


삼성전자의 2025년 4분기 실적 발표에 따르면, 회사는 역대 최고 분기 매출인 93.8조 원을 기록하였으며, 이는 전 분기 대비 9% 증가한 수치입니다. 운영 이익은 20.1조 원으로, 운영 이익률은 21.4%로 7.3%포인트 상승했습니다. 전체 연간 매출은 333.6조 원, 연간 운영 이익은 43.6조 원에 달했습니다.

DX(디바이스 경험) 부문에서는 매출이 전 분기 대비 8% 감소했으나, DS(디바이스 솔루션) 부문은 33% 증가하여 HBM 및 기타 고부가가치 제품의 판매 증가로 인해 강한 성과를 보였습니다. SG&A(판매 및 일반 관리비)는 24.2조 원으로, 전 분기 대비 2.9조 원 증가했습니다. R&D(연구 및 개발) 투자도 10.9조 원으로 증가했습니다. 

이러한 실적은 메모리 가격 상승과 외환 변동의 긍정적인 영향을 받았으며, 특히 미국 달러와 주요 통화의 급격한 상승이 약 1.6조 원의 운영 이익에 기여했습니다.